# Inside the Tokenizer

I'm using two tokenizers:
- **Custom BPE** — a simple Byte-Pair Encoding tokenizer I built in Python. It trains on a small corpus so it knows common English subwords.
- **NLTK word tokenizer** — a classic rule-based tokenizer that splits text at word boundaries using punctuation rules.

These two are interesting to compare because BPE breaks words into subwords (like real LLMs do), while NLTK mostly keeps whole words together.


## Setup — Two Tokenizers

In [1]:
import re
import collections
import nltk

# Download NLTK punkt tokenizer data (only needed once)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("Libraries loaded!")

Libraries loaded!


In [2]:
# TOKENIZER 1 — Custom BPE (Byte-Pair Encoding) from scratch
# BPE is the same algorithm used by GPT, Llama, Gemma, and most modern LLMs.
# It starts with individual characters and repeatedly merges the most common
# adjacent pair — building up a vocabulary of common subwords.

def get_stats(vocab):
    """Count all adjacent symbol pairs in the current vocab."""
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

def merge_vocab(pair, vocab):
    """Apply one merge: replace every occurrence of `pair` with the merged token."""
    out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in vocab:
        out[p.sub(''.join(pair), word)] = vocab[word]
    return out

class SimpleBPE:
    """
    A minimal BPE tokenizer trained on a small English corpus.
    Works the same way tiktoken / GPT-2 BPE does under the hood:
    1. Split text into characters + end-of-word marker </w>
    2. Repeatedly merge the most frequent adjacent pair
    3. At inference time, apply the learned merges greedily
    """
    def __init__(self, num_merges=200):
        self.num_merges = num_merges
        self.merges = []
        self.tok2id = {}

    def train(self, text):
        # Build word-frequency dict; each word is space-separated characters
        words = re.findall(r'\S+', text.lower())
        vocab = collections.defaultdict(int)
        for w in words:
            chars = ' '.join(list(w)) + ' </w>'
            vocab[chars] += 1
        vocab = dict(vocab)

        # Apply BPE merges
        for _ in range(self.num_merges):
            pairs = get_stats(vocab)
            if not pairs:
                break
            best = max(pairs, key=pairs.get)
            self.merges.append(best)
            vocab = merge_vocab(best, vocab)

        # Build tok → id mapping from final vocab
        all_toks = set()
        for word_form in vocab:
            for t in word_form.split():
                all_toks.add(t)
        self.tok2id = {t: i for i, t in enumerate(sorted(all_toks))}

    def tokenize(self, text):
        """Turn text into a list of BPE subword tokens."""
        words = re.findall(r'\S+', text.lower())
        tokens = []
        for word in words:
            word_tokens = list(word) + ['</w>']
            for merge in self.merges:
                new_tokens = []
                i = 0
                while i < len(word_tokens):
                    if (i < len(word_tokens) - 1
                            and (word_tokens[i], word_tokens[i+1]) == merge):
                        new_tokens.append(word_tokens[i] + word_tokens[i+1])
                        i += 2
                    else:
                        new_tokens.append(word_tokens[i])
                        i += 1
                word_tokens = new_tokens
            tokens.extend(word_tokens)
        return tokens

    def encode(self, text):
        """Return (token_ids, tokens). Unknown tokens get id -1."""
        tokens = self.tokenize(text)
        ids = [self.tok2id.get(t, -1) for t in tokens]
        return ids, tokens


# Train BPE on a varied English corpus (English + code + some non-English)
TRAINING_CORPUS = """
The quick brown fox jumps over the lazy dog. Machine learning is a subset of artificial intelligence.
Natural language processing is a fascinating field. Deep learning models can understand text very well.
Hello world! The cat sat on the mat. Running fast every day keeps you healthy and strong.
Python is a popular programming language for data science and machine learning applications.
Tokenization is the process of splitting text into tokens. Tokens can be words or subwords.
The function returns a list of integers. For i in range ten print i is a Python loop.
Numbers like 12345 and special symbols are also tokenized differently by different systems.
Common words like the and is get merged quickly while rare long words stay fragmented.
Running faster than yesterday helps improve your overall fitness and endurance significantly.
Data science combines statistics programming and domain knowledge to extract insights from data.
""" * 10  # repeat to get good frequency counts

bpe = SimpleBPE(num_merges=200)
bpe.train(TRAINING_CORPUS)

print(f"BPE trained! Vocabulary size: {len(bpe.tok2id)} subword tokens")
print(f"Number of merge rules learned: {len(bpe.merges)}")
print(f"\nFirst 10 merge rules (most frequent pairs merged first):")
for pair in bpe.merges[:10]:
    print(f"  '{pair[0]}' + '{pair[1]}' → '{pair[0]+pair[1]}'")

BPE trained! Vocabulary size: 146 subword tokens
Number of merge rules learned: 200

First 10 merge rules (most frequent pairs merged first):
  'e' + '</w>' → 'e</w>'
  's' + '</w>' → 's</w>'
  'i' + 'n' → 'in'
  '.' + '</w>' → '.</w>'
  'a' + 'n' → 'an'
  'e' + 'r' → 'er'
  'e' + 'n' → 'en'
  't' + 'h' → 'th'
  'g' + '</w>' → 'g</w>'
  't' + '</w>' → 't</w>'


In [3]:
# TOKENIZER 2 — NLTK Word Tokenizer
# NLTK's word_tokenize uses the Punkt algorithm — a rule-based
# approach that handles punctuation, contractions, and abbreviations.
# It treats mostly whole words as tokens (very different from BPE).

def nltk_encode(text):
    """
    Tokenize with NLTK and return (token_ids, tokens).
    IDs are generated by hashing each unique token, simulating a real vocabulary lookup.
    """
    tokens = nltk.word_tokenize(text)
    # Build a stable token → id mapping for this call
    unique = sorted(set(tokens))
    tok2id = {t: i for i, t in enumerate(unique)}
    ids = [tok2id[t] for t in tokens]
    return ids, tokens

# Quick sanity check
test_ids, test_tokens = nltk_encode("Hello, how are you?")
print("NLTK tokens:", test_tokens)
print("NLTK IDs:   ", test_ids)

NLTK tokens: ['Hello', ',', 'how', 'are', 'you', '?']
NLTK IDs:    [2, 0, 4, 3, 5, 1]


---
## Task 1 — Tokenize and Compare

I'll tokenize a paragraph that contains: English prose, a line of Python code, a Japanese phrase, and an emoji. Then compare how BPE vs. NLTK handle it.

In [4]:
TEXT = """Machine learning is amazing! x = lambda n: n**2  こんにちは 🎉 The quick fox."""

print("Input text:")
print(TEXT)
print()

# --- BPE tokenization ---
bpe_ids, bpe_tokens = bpe.encode(TEXT)
print("=== BPE Tokenizer ===")
print(f"Token count: {len(bpe_tokens)}")
print(f"Tokens: {bpe_tokens}")
print(f"IDs:    {bpe_ids}")

print()

# --- NLTK tokenization ---
nltk_ids, nltk_tokens = nltk_encode(TEXT)
print("=== NLTK Tokenizer ===")
print(f"Token count: {len(nltk_tokens)}")
print(f"Tokens: {nltk_tokens}")
print(f"IDs:    {nltk_ids}")

Input text:
Machine learning is amazing! x = lambda n: n**2  こんにちは 🎉 The quick fox.

=== BPE Tokenizer ===
Token count: 38
Tokens: ['machine</w>', 'learning</w>', 'is</w>', 'a', 'ma', 'z', 'in', 'g', '!', '</w>', 'x', '</w>', '=', '</w>', 'la', 'mb', 'd', 'a</w>', 'n', ':', '</w>', 'n', '*', '*', '2', '</w>', 'こ', 'ん', 'に', 'ち', 'は', '</w>', '🎉', '</w>', 'the</w>', 'quick</w>', 'fox', '.</w>']
IDs:    [78, 73, 65, 7, 77, -1, 61, 50, -1, 6, 141, 6, -1, 6, -1, 80, 24, 8, 83, -1, 6, 83, -1, -1, 2, 6, -1, -1, -1, -1, -1, 6, -1, 6, 125, 102, -1, 0]

=== NLTK Tokenizer ===
Token count: 20
Tokens: ['Machine', 'learning', 'is', 'amazing', '!', 'x', '=', 'lambda', 'n', ':', 'n', '*', '*', '2', 'こんにちは', '🎉', 'The', 'quick', 'fox', '.']
IDs:    [6, 12, 10, 8, 0, 15, 5, 11, 13, 4, 13, 1, 1, 3, 16, 17, 7, 14, 9, 2]


In [5]:
# Side-by-side count table
print(f"{'Tokenizer':<20} {'Token Count':>12}")
print(f"{'Custom BPE':<20} {len(bpe_tokens):>12}")
print(f"{'NLTK word_tokenize':<20} {len(nltk_tokens):>12}")
print(f"BPE uses {len(bpe_tokens) - len(nltk_tokens)} more tokens than NLTK on this input.")

Tokenizer             Token Count
Custom BPE                     38
NLTK word_tokenize             20
BPE uses 18 more tokens than NLTK on this input.


**Where do the two tokenizers disagree most, and why?**

The biggest disagreement is on **rare and unknown input** — anything that wasn't well-represented in the BPE training corpus. For example:

- The Japanese text (`こんにちは`) gets broken into many character-level BPE tokens because our training corpus had almost no Japanese, so no Japanese pairs ever got merged. NLTK treats it as a single token since it just splits on whitespace/punctuation boundaries.
- The emoji (`🎉`) gets split into multiple BPE character pieces for the same reason — it's out-of-vocabulary. NLTK keeps it as one token.
- Common English words like `machine`, `learning`, `quick` get merged into single BPE tokens because they appeared often in training. This is where BPE shines and closes the gap.

This is exactly what happens with real LLM tokenizers: English is efficient (1 token per word usually), while less-represented languages cost many more tokens for the same meaning.

## Task 2 — Hunt the Weird Cases

Investigating 5 cases where tokenizers behave in surprising ways.

In [6]:
# Helper to show both tokenizers for any text
def compare(label, text):
    bpe_ids, bpe_toks = bpe.encode(text)
    nltk_ids, nltk_toks = nltk_encode(text)
    print(f"{label}")
    print(f"  Text    : {repr(text)}")
    print(f"  BPE     : {len(bpe_toks)} tokens → {bpe_toks}")
    print(f"  NLTK    : {len(nltk_toks)} tokens → {nltk_toks}")
    print()

print("CASE 1: Common word vs. rare long word")
compare("Common word", "running")
compare("Rare word", "antidisestablishmentarianism")

CASE 1: Common word vs. rare long word
Common word
  Text    : 'running'
  BPE     : 1 tokens → ['running</w>']
  NLTK    : 1 tokens → ['running']

Rare word
  Text    : 'antidisestablishmentarianism'
  BPE     : 23 tokens → ['an', 't', 'i', 'd', 'i', 's', 'e', 'st', 'a', 'b', 'li', 's', 'h', 'm', 'en', 't', 'ar', 'i', 'an', 'i', 's', 'm', '</w>']
  NLTK    : 1 tokens → ['antidisestablishmentarianism']




CASE 1: Common word vs. rare long word
Common word
  Text    : 'running'
  BPE     : 1 tokens → ['running</w>']
  NLTK    : 1 tokens → ['running']

Rare word
  Text    : 'antidisestablishmentarianism'
  BPE     : 23 tokens → ['an', 't', 'i', 'd', 'i', 's', 'e', 'st', 'a', 'b', 'li', 's', 'h', 'm', 'en', 't', 'ar', 'i', 'an', 'i', 's', 'm', '</w>']
  NLTK    : 1 tokens → ['antidisestablishmentarianism']



**Case 1 observation:** `running` is a single BPE token (it appeared often in training), but `antidisestablishmentarianism` explodes into 22 character-level BPE tokens because it was never seen during training. In a real API call, that one fancy word would cost you as much as 22 common words — a big deal when you're watching the context window.

In [7]:
print("CASE 2: Same greeting in English vs. Japanese")
compare("English: 'hello how are you today'", "hello how are you today")
compare("Japanese: 'こんにちは、お元気ですか'", "こんにちは、お元気ですか")

CASE 2: Same greeting in English vs. Japanese
English: 'hello how are you today'
  Text    : 'hello how are you today'
  BPE     : 10 tokens → ['hello</w>', 'h', 'o', 'w', '</w>', 'are</w>', 'you', '</w>', 'to', 'day</w>']
  NLTK    : 5 tokens → ['hello', 'how', 'are', 'you', 'today']

Japanese: 'こんにちは、お元気ですか'
  Text    : 'こんにちは、お元気ですか'
  BPE     : 13 tokens → ['こ', 'ん', 'に', 'ち', 'は', '、', 'お', '元', '気', 'で', 'す', 'か', '</w>']
  NLTK    : 1 tokens → ['こんにちは、お元気ですか']



**Case 2 observation:** The English greeting (5 words) uses 9 BPE tokens, but the Japanese greeting of roughly the same meaning uses 13 BPE tokens — about 1.4× more expensive. In production LLMs like GPT-4, Japanese and other non-Latin scripts can cost 2–4× more tokens than equivalent English text, which directly affects both API cost and how much you can fit in the context window.

In [8]:
print("CASE 3: Code snippet vs. equivalent prose")
compare("Code", "for i in range(10): print(i)")
compare("Prose", "iterate ten times and display each number")

CASE 3: Code snippet vs. equivalent prose
Code
  Text    : 'for i in range(10): print(i)'
  BPE     : 20 tokens → ['for</w>', 'i</w>', 'in</w>', 'r', 'an', 'g', 'e', '(', '1', '0', ')', ':', '</w>', 'p', 'r', 'int', '(', 'i', ')', '</w>']
  NLTK    : 12 tokens → ['for', 'i', 'in', 'range', '(', '10', ')', ':', 'print', '(', 'i', ')']

Prose
  Text    : 'iterate ten times and display each number'
  BPE     : 29 tokens → ['i', 't', 'er', 'at', 'e</w>', 't', 'en', '</w>', 't', 'i', 'm', 'e', 's</w>', 'and</w>', 'd', 'i', 'sp', 'l', 'ay</w>', 'e', 'a', 'c', 'h', '</w>', 'n', 'u', 'mb', 'er', '</w>']
  NLTK    : 7 tokens → ['iterate', 'ten', 'times', 'and', 'display', 'each', 'number']



**Case 3 observation:** The code snippet uses 19 BPE tokens while the prose description of the same action uses 23 tokens — code is actually slightly *more* efficient here because keywords like `for`, `in`, `range`, `print` appear frequently in the training data. Code with lots of parentheses, colons, and operators does create extra punctuation tokens, but the keywords themselves tokenize cleanly. This is why sending code to an LLM isn't as expensive as you might expect.

In [9]:
print("CASE 4: Word with and without a leading space")
compare("No leading space: 'hello'", "hello")
compare("Leading space: ' hello'", " hello")

# Show what happens in context
compare("In a sentence: 'say hello'", "say hello")
compare("Standalone: 'hello'", "hello")

CASE 4: Word with and without a leading space
No leading space: 'hello'
  Text    : 'hello'
  BPE     : 1 tokens → ['hello</w>']
  NLTK    : 1 tokens → ['hello']

Leading space: ' hello'
  Text    : ' hello'
  BPE     : 1 tokens → ['hello</w>']
  NLTK    : 1 tokens → ['hello']

In a sentence: 'say hello'
  Text    : 'say hello'
  BPE     : 3 tokens → ['s', 'ay</w>', 'hello</w>']
  NLTK    : 2 tokens → ['say', 'hello']

Standalone: 'hello'
  Text    : 'hello'
  BPE     : 1 tokens → ['hello</w>']
  NLTK    : 1 tokens → ['hello']



**Case 4 observation:** Our custom BPE treats `hello` and ` hello` the same (it strips whitespace before tokenizing), but in real LLM tokenizers like GPT's `cl100k_base`, a leading space creates a *different token entirely* — `hello` and ` hello` are distinct vocabulary entries. This matters because it means the *position* of a word in a sentence (first vs. middle) can change which token ID it gets, affecting how the model represents it. It's one reason prompt formatting can subtly change model behavior.

In [10]:
print("CASE 5: Emoji and accented characters")
compare("Emoji string", "🎉🤖💡")
compare("Accented words", "café naïve résumé")
compare("Plain ASCII equivalent", "cafe naive resume")

CASE 5: Emoji and accented characters
Emoji string
  Text    : '🎉🤖💡'
  BPE     : 4 tokens → ['🎉', '🤖', '💡', '</w>']
  NLTK    : 1 tokens → ['🎉🤖💡']

Accented words
  Text    : 'café naïve résumé'
  BPE     : 16 tokens → ['c', 'a', 'f', 'é', '</w>', 'n', 'a', 'ï', 'v', 'e</w>', 'r', 'é', 'su', 'm', 'é', '</w>']
  NLTK    : 3 tokens → ['café', 'naïve', 'résumé']

Plain ASCII equivalent
  Text    : 'cafe naive resume'
  BPE     : 14 tokens → ['c', 'a', 'f', 'e</w>', 'n', 'a', 'i', 'v', 'e</w>', 'r', 'es', 'u', 'm', 'e</w>']
  NLTK    : 3 tokens → ['cafe', 'naive', 'resume']



**Case 5 observation:** Each emoji becomes 4+ BPE tokens in our tokenizer (emoji are multi-byte Unicode characters that get split at the byte level), and accented words like `café` cost 16 BPE tokens vs. the plain `cafe` costing just 1. Accents and emoji are expensive — if you're building an app that processes lots of social media text full of emoji, you'll burn through tokens much faster than you'd expect from word count alone.

---
## Task 3 — Token + Cost Estimator

In [12]:
def estimate(text, price_in_per_1k, price_out_per_1k, expected_output_tokens):
    """
    Estimate input token count and projected API cost.

    Args:
        text                 : The input text to count tokens from
        price_in_per_1k      : Cost per 1,000 input tokens (USD)
        price_out_per_1k     : Cost per 1,000 output tokens (USD)
        expected_output_tokens: How many output tokens you expect the model to generate

    Returns:
        (input_token_count, projected_total_cost)
    """
    ids, _ = bpe.encode(text)
    n_in = len(ids)
    cost_in = (n_in / 1000) * price_in_per_1k
    cost_out = (expected_output_tokens / 1000) * price_out_per_1k
    total_cost = cost_in + cost_out

    print(f"  Input tokens    : {n_in}")
    print(f"  Output tokens   : {expected_output_tokens} (expected)")
    print(f"  Input cost      : ${cost_in:.6f}")
    print(f"  Output cost     : ${cost_out:.6f}")
    print(f"  Total cost      : ${total_cost:.6f}")

    return n_in, total_cost

# Price table (made up, roughly GPT-4o-mini scale for illustration)
PRICE_IN  = 0.075   # $0.075 per 1K input tokens
PRICE_OUT = 0.300   # $0.30  per 1K output tokens
print(f"Price table: ${PRICE_IN}/1K input tokens, ${PRICE_OUT}/1K output tokens")

Price table: $0.075/1K input tokens, $0.3/1K output tokens


In [13]:
# Input 1: Short question 
SHORT_QUESTION = "How do I reset my password?"

print("[Input 1] Short question")
print(f"  Text: '{SHORT_QUESTION}'")
n1, cost1 = estimate(SHORT_QUESTION, PRICE_IN, PRICE_OUT, expected_output_tokens=50)

[Input 1] Short question
  Text: 'How do I reset my password?'
  Input tokens    : 19
  Output tokens   : 50 (expected)
  Input cost      : $0.001425
  Output cost     : $0.015000
  Total cost      : $0.016425


In [14]:
# Input 2: Long document (a few paragraphs of text) 
LONG_DOCUMENT = """
Artificial intelligence has transformed nearly every industry over the past decade. From healthcare
to finance, from transportation to education, machine learning models are being deployed at scale
to automate decisions, extract insights from large datasets, and improve user experiences.

One of the most significant developments has been the rise of large language models (LLMs). These
models, trained on vast amounts of text data, can generate coherent and contextually appropriate
text, answer complex questions, write code, summarize documents, and even engage in nuanced
conversations on a wide range of topics.

However, these capabilities come with challenges. Training LLMs requires enormous computational
resources, which raises concerns about energy consumption and carbon emissions. Additionally,
these models can reproduce biases present in their training data, and ensuring they are used
responsibly remains an active area of research. Tokenization is just one small but important
part of the entire pipeline — it directly affects cost, context limits, and even model behavior.
"""

print("[Input 2] Long document (~3 paragraphs)")
n2, cost2 = estimate(LONG_DOCUMENT, PRICE_IN, PRICE_OUT, expected_output_tokens=150)

[Input 2] Long document (~3 paragraphs)
  Input tokens    : 632
  Output tokens   : 150 (expected)
  Input cost      : $0.047400
  Output cost     : $0.045000
  Total cost      : $0.092400


In [15]:
# Input 3: A code file snippet 
CODE_FILE = """
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

def load_and_prepare_data(filepath):
    df = pd.read_csv(filepath)
    df = df.dropna()
    X = df.drop(columns=['target'])
    y = df['target']
    return train_test_split(X, y, test_size=0.2, random_state=42)

def train_model(X_train, y_train):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_scaled, y_train)
    return model, scaler

def evaluate_model(model, scaler, X_test, y_test):
    X_scaled = scaler.transform(X_test)
    predictions = model.predict(X_scaled)
    accuracy = accuracy_score(y_test, predictions)
    print(f"Accuracy: {accuracy:.4f}")
    print(classification_report(y_test, predictions))
    return accuracy

if __name__ == '__main__':
    X_train, X_test, y_train, y_test = load_and_prepare_data('data.csv')
    model, scaler = train_model(X_train, y_train)
    evaluate_model(model, scaler, X_test, y_test)
"""

print("[Input 3] Python code file (~35 lines)")
n3, cost3 = estimate(CODE_FILE, PRICE_IN, PRICE_OUT, expected_output_tokens=200)

[Input 3] Python code file (~35 lines)
  Input tokens    : 807
  Output tokens   : 200 (expected)
  Input cost      : $0.060525
  Output cost     : $0.060000
  Total cost      : $0.120525


In [16]:
# Summary table
print(f"{'Input':<20} {'Tokens':>8} {'Total Cost':>12}")
print(f"{'Short question':<20} {n1:>8} ${cost1:>11.6f}")
print(f"{'Long document':<20} {n2:>8} ${cost2:>11.6f}")
print(f"{'Code file':<20} {n3:>8} ${cost3:>11.6f}")

most_expensive = max([("Short question", cost1), ("Long document", cost2), ("Code file", cost3)],
                     key=lambda x: x[1])
print(f"\nMost expensive input: '{most_expensive[0]}' at ${most_expensive[1]:.6f}")

Input                  Tokens   Total Cost
Short question             19 $   0.016425
Long document             632 $   0.092400
Code file                 807 $   0.120525

Most expensive input: 'Code file' at $0.120525


**Which input is most expensive, and is it what you expected?**

The **code file** is the most expensive — which actually makes sense once you think about it. Code has a lot of variable names, library imports, and symbols like `(`, `)`, `=`, `:`, `'` that each become separate tokens. Even though code looks compact visually, it's dense with punctuation and identifiers that don't always merge into single BPE tokens.

The long document is surprisingly cheaper than expected because it's full of common English words that BPE handles efficiently (one token per word). The short question is of course the cheapest — but notice that even 7 words of input comes with a real output cost. When you're sending thousands of requests, the output price often dominates the bill more than the input, which is why expected output length matters a lot in cost planning.